# Tokenizer 分词器
和其它NN 一样, Transformer model 无法直接处理原始文本, 第一步就是将文本输入转换为模型能够理解的数字形式。
使用 分词器 进行完成, 主要作用:
1. 将输入内容拆分为单词、 字词、 各种符号  --> token 
2. 将 token ---> 整数 (mapping rokens to an integer)
3. 添加那些对模型有用的额外数据(special_token)
经过 Tokenizer 后 HF将句子输入其中, 得到一个 字典(参考02.py 代码) 格式的数据

In [21]:
# 由于网络问题, 从本地硬盘中加载对应模型.
import torch
from transformers import AutoTokenizer, AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)
# 核心秘籍：加上 local_files_only=True，强行掐断它的网络请求！
tokenizer = AutoTokenizer.from_pretrained(checkpoint, local_files_only=True)

print("从本地硬盘加载成功！")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.bias   | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


从本地硬盘加载成功！


In [18]:

raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


此时 inputs 为 dict 包含两个 key: input_ids, attention_mask。
input_ids: 两行整数, 行对应一句话
attention_mask: 

In [ ]:
# HF 提供了一个 AutoModel类(最上面已经引入)
# model = AutoModel.from_pretrained(checkpoint)
# Transformer 输出向量通常规模很大, 一般具有三个维度: [bsz, seq_len, h_dim]
# bsz: 批次大小, seq_len: 序列长度, h_dim: 隐藏层的大小(每个模型输入向量的维度)


In [22]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([2, 16, 768])


In [ ]:
print(inputs["input_ids"].shape) # 原句子的维度 [bsz, seq_len]

torch.Size([2, 16])


In [30]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, local_files_only=True)
outputs = model(**inputs)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [32]:
print(outputs.logits.shape)
# 只有两句话和两个标签, 模型最终输出结果为 2*2 的矩阵

torch.Size([2, 2])


In [34]:
print(outputs.logits)
#  return 2*2 shape matrix :[[], []]

tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)


In [36]:
# outputs.logits 返回的矩阵为两个句子的得分情况, [-1.5607,  1.6123]
# [4.1692, -3.3464], 这个数值并不是概率值, 而是模型最后一层输出的原始、未经归一化的数值(逻辑值)
# 逻辑值 转换为概率值, 需要通过Softmax 进行处理
# 训练过程中, 损失函数通常会将模型的最后一层输出值（如 SoftMax 的输出）与实际的损失函数（如交叉熵）结合起来使用.
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

tensor([[4.0195e-02, 9.5980e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)


In [ ]:
# 查看model 中的一些配置属性
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}

# 下面为models 的一些内容

In [42]:
# local_files_only=True 本地加载模型.
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-cased", local_files_only=True)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [43]:
# 查看模型config属性
print(model.config)

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.12.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}



In [44]:
from transformers import BertModel

model = BertModel.from_pretrained("bert-base-cased", local_files_only=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [61]:
# 使用 tokenizer 将文本拆分成 token
# encoded_input = tokenizer("Hello, I'm a single sentence!")
encoded_input = tokenizer("Hello, I'm a single sentence!",return_tensors="pt")

# return_tensors="pt" 如果不写入这个, return 一个 list, 并不是返回Tensor 
# .pt, 将list 返回为 Torch数据类型(Tensor)
print(encoded_input)


{'input_ids': tensor([[ 101, 7592, 1010, 1045, 1005, 1049, 1037, 2309, 6251,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [ ]:
# print(len(encoded_input["attention_mask"]))

print(encoded_input["input_ids"].shape)

'''
input_ids: tokens 的数值
token_type_ids: (双句子任务中(参考m02.py)), 输入内容中的哪一部分属于句子 A, 哪一部分属于句子 B
attention_mask: 需要关注的 Token.
'''

torch.Size([1, 11])


In [ ]:
tokenizer.decode(encoded_input["input_ids"])

["[CLS] hello, i ' m a single sentence! [SEP]"]

In [ ]:
# 两个特殊的 Token [CLS], [SEP]. 

# 观察到两个句子长度并不相同, Tokenizer 需要进行 PAD进行对齐.
# 
encoded_input = tokenizer("How are you?", "I'm fine, thank you!", return_tensors="pt")


print(encoded_input)
# HF 官网文档给出：
'''
{'input_ids': tensor([[  101,  1731,  1132,  1128,   136,   102],
         [  101,  1045,  1005,  1049,  2503,   117,  5763,  1128,   136,   102]]), 
 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1],
         [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
'''

# 运行后为:
'''
{'input_ids': tensor([[ 101, 2129, 2024, 2017, 1029,  102, 1045, 1005, 1049, 2986, 1010, 4067,
         2017,  999,  102]]), 
  'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 
  'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
'''
# (发现并不相同 ) 底层处理机制不同(不进行细查)
# token 101[CLS]  102[SEP]


{'input_ids': tensor([[ 101, 2129, 2024, 2017, 1029,  102, 1045, 1005, 1049, 2986, 1010, 4067,
         2017,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


"\n{'input_ids': tensor([[ 101, 2129, 2024, 2017, 1029,  102, 1045, 1005, 1049, 2986, 1010, 4067,\n         2017,  999,  102]]), \n  'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), \n  'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}\n"

In [ ]:
# 查看此时shape 发现[1, 15] 将上述两句话划分为 一个长为 15 的seq
print(encoded_input["input_ids"].shape) 


torch.Size([1, 15])


In [77]:
text = ["How are you?", 
        "I'm fine, thank you!"
]
new_encoded_input = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
print(new_encoded_input) 


{'input_ids': tensor([[ 101, 2129, 2024, 2017, 1029,  102,    0,    0,    0,    0],
        [ 101, 1045, 1005, 1049, 2986, 1010, 4067, 2017,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [ ]:
print(new_encoded_input["input_ids"].shape) 
# 返回 [2, 10] 两句话 长度为 10, seq长度小于10 进行 0 填充保证对齐


torch.Size([2, 10])


In [84]:
# 使用 Truncating  进行截断
# 
encoded_input = tokenizer(
    "This is a very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very very long sentence.",
    truncation=True,
    max_length = 10,
)
print(encoded_input["input_ids"])
print(encoded_input)
# 对比 max_length = 10 是否注释可以查看 seq 是否被截断.


[101, 2023, 2003, 1037, 2200, 2200, 2200, 2200, 2200, 102]
{'input_ids': [101, 2023, 2003, 1037, 2200, 2200, 2200, 2200, 2200, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


分词有很多种类
Word-based 基于文字的分词, 基于单词进行分析(vocab_size大, 且 dog dogs 在处理上不同, 模型无法判断dog dogs是相似的)

Character-based 基于字符的分词 (将每个单词拆分为字符特征)

Subword tokenization 字词分词 (那些经常出现的单词不应被拆分成更小的子词，而那些不常见的单词则应被分解成有意义的子词。)

In [86]:
tokenized_text = "Jim Henson was a puppeteer".split()
print(tokenized_text)

['Jim', 'Henson', 'was', 'a', 'puppeteer']


In [87]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-cased", local_files_only=True)

In [88]:
tokenizer("Using a Transformer network is simple")

{'input_ids': [2, 1, 1, 1, 1, 1, 1, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

In [89]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", local_files_only=True)

In [90]:
tokenizer("Using a Transformer network is simple")

{'input_ids': [2, 1, 1, 1, 1, 1, 1, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

将文本转换为数字的过程称为编码, 编码有两个步骤: 1. 文本分词 2. 将分词后的结果转换为响应数字标识.


In [ ]:
# 分词器的中间过程  seq --> 分词、 embedding --> token (dict数据)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", local_files_only=True)
sequence = "Using a Transformer network is simple"
tokens = tokenizer.tokenize(sequence)

print(tokens)

['Using', 'a', 'Trans', '##former', 'network', 'is', 'simple']


In [97]:
ids = tokenizer.convert_tokens_to_ids(tokens)

print(ids)

[7993, 170, 13809, 23763, 2443, 1110, 3014]


In [ ]:
# 省略中间过程, 直接将 seq --> token 
ids_new = tokenizer(sequence,)
print(ids_new)

{'input_ids': [101, 7993, 170, 13809, 23763, 2443, 1110, 3014, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
# 解码过程 由 token(ids) --> seq
decoded_string = tokenizer.decode([7993, 170, 11303, 1200, 2443, 1110, 3014])
print(decoded_string)
# decode 方法不仅会将索引转换回对应的标记, 还将属于同一个单词的各个标记组合在一起，从而形成可读的句子。



Using a transformer network is simple
